In [1]:
"""
Personalized (subject-specific) evaluation with EMULATED SESSIONS and
5-FOLD CROSS-VALIDATION using separate TRAIN / VALIDATION / TEST folds.

Model: LSTM / GRU / CNN encoder. With USE_FEATURES = True, two scalar gait
features (cadence, duration variability) are concatenated after the encoder
(Feature Concat). With USE_FEATURES = False the model sees only the sequence
(Sequence Only).

WHY: the data were recorded as one continuous walk per participant and gait
condition, so there are no real trials or sessions, and neighbouring gait cycles
are highly correlated. Randomly splitting cycles would leak information.

DATA DIVISION (done separately for every participant and gait class):
  1. Cycles are put back in walking order (cycle id from the filename).
  2. They are grouped into emulated SESSIONS of SESSION_LEN consecutive cycles.
  3. GUARD_LEN cycles between every two sessions are discarded (guard band), so
     cycles from different sessions are never neighbours.
  4. Sessions are shuffled once and dealt into N_FOLDS folds (round robin), so
     every fold contains sessions of every gait class.

CROSS-VALIDATION: for fold k = 1..N_FOLDS
  - TEST       = fold k                 (used only for the final score)
  - VALIDATION = fold k+1 (wraps round) (used only for early stopping /
                                         choosing the best epoch)
  - TRAIN      = the other N_FOLDS-2 folds
Every session is used for testing exactly once. Results are reported per fold,
as the mean over folds, and as the pooled accuracy over all test predictions.

Leak-safety of the scalar features: duration variability is computed from the
TRAIN cycles only; the scaler is fit on TRAIN only; cadence is per cycle.
For a given participant the split is identical for all three sensor
configurations and for the USE_FEATURES = True / False runs.
"""
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, GRU, Conv1D, GlobalAveragePooling1D,
                                     Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100
SAMPLING_RATE = 100.0
VALID_LABELS = ['back', 'front', 'normal', 'side']

USE_FEATURES = True        # <-- True = Feature Concat, False = Sequence Only

IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1', 'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7',
    'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32
PATIENCE = 5
MODEL_TYPES = ['lstm', 'gru', 'cnn']

# Session / fold settings
SESSION_LEN = 12     # consecutive cycles per emulated session
GUARD_LEN = 6        # cycles discarded between consecutive sessions
N_FOLDS = 5          # folds: 1 test + 1 validation + (N_FOLDS-2) train
FOLD_SEED = 42       # controls how sessions are shuffled into folds

RESULTS_DIR = 'gait_hardware_paper_results'
tag = 'featconcat' if USE_FEATURES else 'seqonly_concat'
FOLDS_CSV = os.path.join(RESULTS_DIR, f'personalized_{tag}_kfold_perfold.csv')     # one row per fold
SUMMARY_CSV = os.path.join(RESULTS_DIR, f'personalized_{tag}_kfold_summary.csv')   # per subject/config/model


def get_subject_id(file_path):
    return os.path.relpath(file_path, ROOT_DIR).split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw
    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                fp = os.path.join(root, filename)
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cid = raw_id_part.split('_')[-1]
                if label not in VALID_LABELS or not cid.isdigit():
                    continue
                try:
                    df = pd.read_csv(fp)
                except Exception:
                    continue
                raw.append((df, label, get_subject_id(fp), int(cid)))
    return raw


def prepare_subject_matrix(subject_rows, feature_cols):
    X, y, nrows, cids = [], [], [], []
    for df, label, _, cid in subject_rows:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        nrows.append(vals.shape[0])
        X.append(resample_cycle(vals))
        y.append(label)
        cids.append(cid)
    if len(X) == 0:
        return None, None, None, None
    return np.array(X), np.array(y), np.array(nrows), np.array(cids)


def make_sessions(ordered_idx, session_len, guard_len):
    """Cut temporally ordered indices into sessions separated by guard bands.
    Layout: [session][guard][session][guard]...[session].
    A final partial session is kept if it has at least half of session_len,
    otherwise its cycles are discarded with the guard cycles."""
    sessions, guard = [], []
    i, n = 0, len(ordered_idx)
    while i < n:
        sess = ordered_idx[i:i + session_len]
        if len(sess) >= session_len / 2:
            sessions.append(sess)
        else:
            guard.extend(sess)
        i += session_len
        guard.extend(ordered_idx[i:i + guard_len])   # guard band after the session
        i += guard_len
    return sessions, guard


def assign_folds(y_int, cycle_ids, session_len, guard_len, n_folds, rng):
    """For each class: order by cycle id, build sessions with guard bands,
    shuffle the sessions and deal them into n_folds folds (round robin).
    Returns fold_of (fold number per cycle, -1 = guard/discarded) and the
    smallest number of sessions found in any class."""
    fold_of = np.full(len(y_int), -1, dtype=int)
    min_sessions = None
    for c in np.unique(y_int):
        idx = np.where(y_int == c)[0]
        idx = idx[np.argsort(cycle_ids[idx], kind='stable')]   # temporal order
        sessions, _ = make_sessions(idx, session_len, guard_len)
        k = len(sessions)
        min_sessions = k if min_sessions is None else min(min_sessions, k)
        order = rng.permutation(k)
        for pos, s_i in enumerate(order):
            fold_of[sessions[s_i]] = pos % n_folds
    return fold_of, min_sessions


def build_model(model_type, seq_len, num_features, num_scalars, num_classes, use_features):
    """Functional model. If use_features, a second scalar input is concatenated
    after the sequence encoder; otherwise it's a single-input model."""
    seq_in = Input(shape=(seq_len, num_features), name='sequence')
    if model_type == 'lstm':
        x = LSTM(UNITS, name='LSTM')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'gru':
        x = GRU(UNITS, name='GRU')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'cnn':
        x = Conv1D(64, 5, activation='relu')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
        x = Conv1D(64, 5, activation='relu')(x)
        x = GlobalAveragePooling1D()(x)
        x = Dropout(DROPOUT_RATE)(x)
    else:
        raise ValueError(model_type)

    if use_features:
        scal_in = Input(shape=(num_scalars,), name='scalars')
        x = Concatenate()([x, scal_in])
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=[seq_in, scal_in], outputs=out)
    else:
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=seq_in, outputs=out)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def run():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return
    subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(subjects)} subjects: {subjects}")
    print(f"USE_FEATURES = {USE_FEATURES} | sessions of {SESSION_LEN} cycles, "
          f"guard {GUARD_LEN} cycles, {N_FOLDS} folds (train/val/test)\n")

    fold_rows, summary_rows = [], []

    for subj_i, subj in enumerate(subjects):
        subj_rows = [r for r in raw if r[2] == subj]
        print(f"\n{'='*60}\n=== Subject: {subj} ({len(subj_rows)} cycles) ===\n{'='*60}")

        for cfg_name, cols in CONFIGS.items():
            X, y_str, nrows, cids = prepare_subject_matrix(subj_rows, cols)
            if X is None:
                print(f"  [{cfg_name}] no usable cycles, skipping")
                continue

            le = LabelEncoder()
            y_int = le.fit_transform(y_str)
            num_classes = len(le.classes_)
            y_cat = to_categorical(y_int, num_classes=num_classes)
            n = X.shape[0]

            for c in np.unique(y_int):
                ids_c = cids[y_int == c]
                if len(np.unique(ids_c)) != len(ids_c):
                    print(f"  WARNING [{cfg_name}] duplicate cycle ids in class "
                          f"{le.classes_[c]} -- temporal order may be unreliable")

            # same seed for every config and for both USE_FEATURES runs -> same folds
            rng = np.random.default_rng([FOLD_SEED, subj_i])
            fold_of, min_sessions = assign_folds(y_int, cids, SESSION_LEN, GUARD_LEN, N_FOLDS, rng)
            if min_sessions < N_FOLDS or num_classes < 2:
                print(f"  [{cfg_name}] a class has only {min_sessions} sessions "
                      f"(need {N_FOLDS}), skipping")
                continue
            n_guard = int((fold_of < 0).sum())
            print(f"  [{cfg_name}] cycles used={n - n_guard}, guard(discarded)={n_guard}")

            seq_len, num_features = X.shape[1], X.shape[2]
            # out-of-fold test predictions, one array per model
            oof = {m: np.full(n, -1, dtype=int) for m in MODEL_TYPES}

            for k in range(N_FOLDS):
                te_idx = np.where(fold_of == k)[0]
                va_idx = np.where(fold_of == (k + 1) % N_FOLDS)[0]
                tr_idx = np.where((fold_of >= 0) & (fold_of != k)
                                  & (fold_of != (k + 1) % N_FOLDS))[0]

                # scalar features: variability from TRAIN only, scaler fit on TRAIN only
                cadence = SAMPLING_RATE / nrows
                dur_std_train = float(np.std(nrows[tr_idx]))
                scalar = np.column_stack([cadence, np.full(n, dur_std_train)])
                num_scalars = scalar.shape[1]
                if USE_FEATURES:
                    scalar = StandardScaler().fit(scalar[tr_idx]).transform(scalar)

                def inputs(idx):
                    return [X[idx], scalar[idx]] if USE_FEATURES else X[idx]

                for mtype in MODEL_TYPES:
                    tf.keras.backend.clear_session()
                    model = build_model(mtype, seq_len, num_features, num_scalars,
                                        num_classes, USE_FEATURES)
                    es = EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                       restore_best_weights=True, verbose=0)
                    hist = model.fit(inputs(tr_idx), y_cat[tr_idx],
                                     validation_data=(inputs(va_idx), y_cat[va_idx]),
                                     epochs=EPOCHS, batch_size=BATCH_SIZE,
                                     callbacks=[es], verbose=0)
                    best_epoch = int(np.argmin(hist.history['val_loss'])) + 1

                    pred = np.argmax(model.predict(inputs(te_idx), verbose=0), axis=1)
                    oof[mtype][te_idx] = pred
                    y_te = y_int[te_idx]
                    acc = accuracy_score(y_te, pred)
                    print(f"  [{cfg_name:12s} {mtype.upper():4s}] fold {k+1}: "
                          f"train={len(tr_idx):4d} val={len(va_idx):3d} test={len(te_idx):3d} "
                          f"best_epoch={best_epoch:2d}  acc={acc:.4f}")
                    fold_rows.append({
                        'subject': subj, 'config': cfg_name, 'model': mtype,
                        'use_features': USE_FEATURES, 'fold': k + 1,
                        'n_train': len(tr_idx), 'n_val': len(va_idx), 'n_test': len(te_idx),
                        'n_guard': n_guard, 'best_epoch': best_epoch,
                        'accuracy': round(acc, 4),
                        'macro_f1': round(f1_score(y_te, pred, average='macro', zero_division=0), 4),
                        'weighted_f1': round(f1_score(y_te, pred, average='weighted', zero_division=0), 4),
                    })

            # per subject/config/model summary
            used = fold_of >= 0
            for mtype in MODEL_TYPES:
                fa = [r['accuracy'] for r in fold_rows
                      if r['subject'] == subj and r['config'] == cfg_name and r['model'] == mtype]
                y_all, p_all = y_int[used], oof[mtype][used]
                summary_rows.append({
                    'subject': subj, 'config': cfg_name, 'model': mtype,
                    'use_features': USE_FEATURES,
                    'mean_fold_accuracy': round(float(np.mean(fa)), 4),
                    'std_fold_accuracy': round(float(np.std(fa)), 4),
                    'pooled_accuracy': round(accuracy_score(y_all, p_all), 4),
                    'pooled_macro_f1': round(f1_score(y_all, p_all, average='macro', zero_division=0), 4),
                    'pooled_weighted_f1': round(f1_score(y_all, p_all, average='weighted', zero_division=0), 4),
                    'n_cycles_used': int(used.sum()), 'n_guard': n_guard,
                })

    if not summary_rows:
        print("\nNo results produced.")
        return
    os.makedirs(RESULTS_DIR, exist_ok=True)
    pd.DataFrame(fold_rows).to_csv(FOLDS_CSV, index=False)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(SUMMARY_CSV, index=False)
    print(f"\n\nWrote {len(fold_rows)} fold rows to {FOLDS_CSV}")
    print(f"Wrote {len(summary)} summary rows to {SUMMARY_CSV}")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(summary.pivot_table(index='subject', columns=['config', 'model'],
                                  values='mean_fold_accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run()

2026-10-05 16:33:25.759175: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791232405.774806 1869996 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791232405.779972 1869996 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791232405.793842 1869996 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791232405.793855 1869996 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791232405.793857 1869996 computation_placer.cc:177] computation placer alr

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']
USE_FEATURES = True | sessions of 12 cycles, guard 6 cycles, 5 folds (train/val/test)


=== Subject: Andy_Dynamic (424 cycles) ===
  [IMU] cycles used=288, guard(discarded)=136


I0000 00:00:1791232429.066215 1869996 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 985 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1791232429.071672 1869996 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46310 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1791232429.073567 1869996 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46412 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1791232429.075311 1869996 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 46412 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
I0000 00:00:1791232431.216169 1946731 cuda_dnn.cc:529] Loaded cuDNN version 90701


  [IMU          LSTM] fold 1: train= 144 val= 60 test= 84 best_epoch=24  acc=0.9524
  [IMU          GRU ] fold 1: train= 144 val= 60 test= 84 best_epoch=25  acc=0.9048


I0000 00:00:1791232455.360467 1946723 service.cc:152] XLA service 0x1a4046f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791232455.360529 1946723 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791232455.360565 1946723 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791232455.360568 1946723 service.cc:160]   StreamExecutor device (2): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1791232455.360569 1946723 service.cc:160]   StreamExecutor device (3): NVIDIA RTX A6000, Compute Capability 8.6
2026-10-05 16:34:15.388467: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791232457.375908 1946723 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process

  [IMU          CNN ] fold 1: train= 144 val= 60 test= 84 best_epoch=25  acc=0.9762
  [IMU          LSTM] fold 2: train= 180 val= 48 test= 60 best_epoch=22  acc=0.8333
  [IMU          GRU ] fold 2: train= 180 val= 48 test= 60 best_epoch=25  acc=0.9833
  [IMU          CNN ] fold 2: train= 180 val= 48 test= 60 best_epoch=24  acc=0.9667
  [IMU          LSTM] fold 3: train= 192 val= 48 test= 48 best_epoch=22  acc=0.8750
  [IMU          GRU ] fold 3: train= 192 val= 48 test= 48 best_epoch=25  acc=0.8958
  [IMU          CNN ] fold 3: train= 192 val= 48 test= 48 best_epoch=13  acc=0.9792
  [IMU          LSTM] fold 4: train= 192 val= 48 test= 48 best_epoch=23  acc=0.7917
  [IMU          GRU ] fold 4: train= 192 val= 48 test= 48 best_epoch=24  acc=0.8542
  [IMU          CNN ] fold 4: train= 192 val= 48 test= 48 best_epoch=23  acc=0.9375
  [IMU          LSTM] fold 5: train= 156 val= 84 test= 48 best_epoch=24  acc=0.8750
  [IMU          GRU ] fold 5: train= 156 val= 84 test= 48 best_epoch=25  acc